# Ticket data preprocessing
This notebook is a thin interface to the shared preprocessing package. Configure a raw CSV/ZIP path or a public Hugging Face dataset with an exact commit. It never reopens evaluation data, overwrites existing outputs, or indexes Qdrant. The repository package must be installed; all configuration is in `PreprocessingSettings` and `.env`. See the main README.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path

from app.core.config import PreprocessingSettings
from app.preprocessing.cli import read_huggingface, read_raw
from app.preprocessing.pipeline import export, preprocess

config = PreprocessingSettings()

In [ ]:
input_path = config.input
hf_dataset = config.hf_dataset
hf_revision = config.hf_revision or ""
if input_path:
    raw, source = read_raw(Path(input_path))
elif hf_dataset:
    raw, source = read_huggingface(hf_dataset, hf_revision)
else:
    raise ValueError("Set PREPROCESS_INPUT or PREPROCESS_HF_DATASET and PREPROCESS_HF_REVISION")

In [ ]:
prepared = preprocess(raw, config, source)
# Aggregate statistics only; no raw ticket samples or evaluation-file reads.
prepared.summary

In [ ]:
destination = config.output_root / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
export(prepared, destination, source)
print(f"Created fresh artifacts at {destination}; no Qdrant writes")